In [6]:
# 신경망 : 데이터에 대한 연산을 수행하는 계층(layor)/모듈(module)로 구성
# torch.nn(from torch import nn) : 신경망을 구성하는 데 필요한 구성요소 제공
# Pytorch의 모든 모듈은 nn.Module를 상속
# 신경망의 역할 : 데이터를 입력받아 특징을 추출하고, 이를 바탕으로 분류·예측·생성 등 다양한 작업을 수행

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [7]:
# 학습을 위한 장치 얻기 (파이토치는 계산을 cpu, gpu가 하기 때문)
device = (
  "cuda"
  if torch.cuda.is_available()
  else "mps"
  if torch.backends.mps.is_available()
  else "cpu"
)

print(f"Using {device} device")

Using cuda device


In [8]:
# nn.Module을 상속해 신경망 초기화
# 모든 클래스는 forward 메서드에 입력 데이터에 대한 연산을 구현(데이터의 흐름을 정의)
# layer : 데이터를 받아 조금씩 변환해서 최종적으로 원하는 형태로 만들어서 출력(변환, 활성화, 정규화 등의 역할 수행)
# 변환 레이어 : nn.Flatten
# 활성 레이어 : nn.ReLU
class NeuralNetwork(nn.Module):
  def __init__(self): # 사용할 layor들을 선언
    super().__init__()
    # 다차원 이미지를 1차원으로 폄.
    # (28 * 28) 이미지 -> 784개의 숫자로
    self.flatten = nn.Flatten() 
    self.linear_relu_stack = nn.Sequential( # layor들을 순서대로 연결 (입력 → Linear → ReLU → Linear → ReLU → Linear → 출력)
      # nn.Linear(입력크기, 출력크기) - 완전연결층, 가중치 학습
      # 784개의 입력을 512의 출력으로 변환
      # 완전 연결층 : 입력의 모든 값이 출력의 모든 값과 연결 (입력 3개고 출력 2개면 3 * 2 = 6개의 연결선)
      # 가중치 : 각 연결선 마다 붙어있는 숫자 (학습하는 동안 수치가 변하면서 최적화되어 예측이 정확해지게 하는 역할)
      # nn.Linear(28*28, 512) : 784개와 512개를 전부 연결하고 각 연결선마다 가중치(784 * 512개의 가중치)가 존재
      nn.Linear(28*28, 512), 
      # 활성화 함수 : 층을 깊게 쌓을 수 있게 함, 음수는 0으로 양수는 그대로 (비선형성 추가 - 복잡한 문제 풀기 가능)
      nn.ReLU(), 
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10),
    ) # 최종 : 784개의 픽셀 값 -> 10개의 정답 클래스로

  def forward(self, x): # 데이터가 레이어를 거치는 순서
    # self.flatten에 ()를 붙이는 것은 .__call__()메서드가 생략되어 있는 것이다.
    # 클래스에 __call__메서드가 있으면 객체에서 직접 호출이 가능하다.
    x = self.flatten(x) # __call__이 자동으로 호출되어 다차원을 1차원으로 변형
    logits = self.linear_relu_stack(x)
    return logits


In [9]:
# GPU 사용하고 싶을 때
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [10]:
X = torch.rand(1, 28, 28, device=device) # 3차원 텐서 만들기, 연산은 device로
logits = model(X)
# model(X)로 시작되는 실행 흐름
# 1. __call__ 자동 호출(__call__메서드는 객체를 함수처럼 호출 가능하게 만든다.)
# NeuralNetwork에 __call__이 없으니 상속받은 부모의 __call__자동 실행(상속받으면 부모 것을 사용 가능하기 때문)
# 2. nn.Module의 __call__메서드에는 forward메서드가 존재하여 호출이 됨.
# 하지만 nn.Module을 상속한 클래스는 반드시 forward를 구현해야 함.
# 3. override된 NeuralNetowrk의 forward실행
# NeuralNetwork의 forward는 해당 클래스에 선언된 레이어들을 정한 순서대로 실행시킨다.

# 모델의 예측 결과를 사람이 알 수 있게 해주는 코드
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([7], device='cuda:0')


In [11]:
input_image = torch.rand(3, 28, 28) # 컬러 이미지 픽셀
print(input_image.size())

torch.Size([3, 28, 28])


In [12]:
flatten = nn.Flatten() # start_dim=1, end_dim=-1이어서 0차원은 평탄화에 포함 X
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [13]:
# nn.Linear(선형 계층) : 저장된 가중치와 편향으로 입력에 선형 변환을 적용
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image) # nn.Sequential()에 nn.Linear를 넣으면 이 과정을 알아서 실행 nn.Linear()만 넣으면 됨.
print(hidden1.size())

torch.Size([3, 20])


In [14]:
# nn.ReLU(비선형 활성화) : 선형에 비선형성을 도입히여 신경망이 다양한 학습을 할 수 있도록 도움
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2446, -0.0343, -0.3754,  0.5985,  0.0086,  0.9246,  0.0702,  0.0579,
         -0.0638,  0.1216, -0.1154,  0.0154, -0.0367,  0.6141,  0.1444,  0.0157,
          0.0088, -0.0628,  0.4118,  0.2987],
        [ 0.1116,  0.3940, -0.1309,  0.5031, -0.0856,  0.4908, -0.1892,  0.1620,
         -0.3969, -0.1029,  0.0183, -0.1085, -0.0998,  0.4837, -0.0811, -0.4540,
         -0.4522,  0.0296,  0.3796,  0.3474],
        [ 0.4346,  0.5574, -0.3793,  0.7912,  0.2411,  0.5184, -0.0557,  0.1467,
         -0.2802, -0.0671, -0.4538, -0.1160, -0.1706,  0.6145, -0.0189, -0.1881,
         -0.4204,  0.0488,  0.3229,  0.2317]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2446, 0.0000, 0.0000, 0.5985, 0.0086, 0.9246, 0.0702, 0.0579, 0.0000,
         0.1216, 0.0000, 0.0154, 0.0000, 0.6141, 0.1444, 0.0157, 0.0088, 0.0000,
         0.4118, 0.2987],
        [0.1116, 0.3940, 0.0000, 0.5031, 0.0000, 0.4908, 0.0000, 0.1620, 0.0000,
         0.0000, 0.0183, 0.0000, 0.0000, 0.4837, 0.00

In [15]:
# Sequential : 데이터의 순전파(forward) 순서를 정해주는 컨테이너
seq_modules = nn.Sequential(
  # 기본적으로 __call__ 메서드가 있는 모든 nn.Module을 상속받은 레이어들은 Sequential에 넣을 수 있다.
  flatten, # 해당 객체는 분리도 가능(분리 시에는 매개변수 전달하지만 Sequential에서는 자동으로 __call호출)
  layer1,
  nn.ReLU(),
  nn.Linear(20, 10)
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

In [16]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"CUDA 버전: {torch.version.cuda}")
print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
print(f"GPU 개수: {torch.cuda.device_count()}")

PyTorch 버전: 2.9.1+cu126
CUDA 사용 가능: True
CUDA 버전: 12.6
GPU 이름: NVIDIA GeForce RTX 3060
GPU 개수: 1
